# WGCNA

# Import Library

In [ ]:
#Install remotes first
# install.packages("remotes")

#Install fgsea via GitHub
# remotes::install_github("ctlab/fgsea", dependencies=FALSE)

In [ ]:
# BiocManager::install("clusterProfiler")

In [ ]:
install.packages("tidyverse")

In [ ]:
library("tidyverse")
options(repr.plot.width=5, repr.plot.height=5)#Default=7x7

#Bioconductor
for (package in c("DEqMS","WGCNA")) {
    if (!requireNamespace("BiocManager", quietly=TRUE))
        install.packages("BiocManager")
    # BiocManager::install(package)
    eval(bquote(library(.(package))))
    print(str_c(package, ": ", as.character(packageVersion(package))))
}
#CRAN
for (package in c("openxlsx", "readxl", "RColorBrewer", "ggrepel")) {
    # install.packages(package)
    eval(bquote(library(.(package))))
    print(str_c(package, ": ", as.character(packageVersion(package))))
}

In [ ]:
# install.packages("car")

# Loading Data

In [ ]:
fileDir <- "./Data/"
fileName <- "pro_data_common_val_Assay.csv"

Path <- str_c(fileDir, fileName)
temp <- read.csv(Path, row.names = 1)
head(temp)
pro_data_val <- temp


fileDir <- "./Data/"
fileName <- "pro_sample_val.csv"

Path <- str_c(fileDir, fileName)
temp <- read.csv(Path)
head(temp)
pro_sample <- temp

fileDir <- "./Data/"
fileName <- "map_OID_Uni_Sym.csv"

Path <- str_c(fileDir, fileName)
temp <- read.csv(Path, row.names = 1)
head(temp)
pro_analyte <- temp

fileDir <- "./Data/"
fileName <- "DEP_list.csv"

path <- file.path(fileDir, fileName)

temp <- read.csv(path,  check.names = FALSE)

DEP_combined <- temp

DEP_Early_vs_Control <- DEP_combined[
  DEP_combined$Contrast == "Early_vs_Control",
]

DEP_Advanced_vs_Control <- DEP_combined[
  DEP_combined$Contrast == "Advanced_vs_Control",
]

DEP_Advanced_vs_Early <- DEP_combined[
  DEP_combined$Contrast == "Advanced_vs_Early",
]

View(DEP_combined)
View(DEP_Early_vs_Control)
View(DEP_Advanced_vs_Control)
View(DEP_Advanced_vs_Early)


# fileDir <- "./Data/"
# fileName <- "target_labels260130.csv"

# Path <- str_c(fileDir, fileName)
# temp <- read.csv(Path)
# head(temp)
# target_labels <- temp

In [ ]:
head(pro_analyte)
str(pro_analyte)

In [ ]:
temp <- pro_analyte %>%
    select(Symbol, UniProt, OlinkID)
# temp$OlinkID <- pro_analyte['OlinkID']
# rownames(temp) <- NULL
head(temp)
pro_analyte <- temp
print(str_c("Unique analytes: ", length(unique(pro_analyte$UniProt))))
print(str_c("Unique annotations: ", length(unique(pro_analyte$Symbol))))

In [ ]:
# fileDir <- "./original/"
# ipynbname <- "code_pancreas"
# fileName <- "PE_R345_AN01_NPX_2025-03-04.csv"

# #original data
# meta_annot <- read_delim(str_c(fileDir,fileName), delim = ";") %>%
#     filter(startsWith(SampleID, "PE_R345"))

# print(str_c("nrow: ", nrow(meta_annot)))
# # print(colnames(bgdDF))
# head(meta_annot)
# # print(bgdDF[ ,1:10])
# print(str_c("Unique analytes: ", length(unique(meta_annot$UniProt))))
# print(str_c("Unique annotations: ", length(unique(meta_annot$SampleID))))

# Create modules

In [ ]:
head(pro_data_val)
dim(pro_data_val)

### power selection

In [ ]:
temp <- pro_data_val

library(WGCNA)

sampleTree <- hclust(dist(temp), method = "average")
plot(sampleTree)

powers <- c(1:10)
sft <- pickSoftThreshold(temp, powerVector = powers, verbose = 5)

net <- blockwiseModules(temp,power = sft$powerEstimate,
                        TOMType = "unsigned", minModuleSize = 30)

#power plotz
plot(sft$fitIndices[,1], -sign(sft$fitIndices[,3])*sft$fitIndices[,2],
     xlab="Soft Threshold (power)", ylab="Scale Free Topology Model Fit, signed R²",
     type="n")
text(sft$fitIndices[,1], -sign(sft$fitIndices[,3])*sft$fitIndices[,2],
     labels=sft$fitIndices[,1], col="red")
abline(h=0.8, col="blue", lty=2)

plot(sft$fitIndices[,1], sft$fitIndices[,5],
     xlab="Soft Threshold (power)", ylab="Mean Connectivity", type="n",
     main=paste("Mean connectivity"))
text(sft$fitIndices[,1], sft$fitIndices[,5], labels=powers, cex=0.8, col="black")

print(str_c("Soft-thresholding power: ", sft$powerEstimate))

using power = 3

### adjacency matrix

~~It takes a long time using adjacency method, so using blockwiseModules method instead~~
using original method

In [ ]:
# # library(WGCNA)
# temp <- pro_data_val

# moduleLabels <- net$colors
# moduleColors <- labels2colors(moduleLabels)

# softPower <- sft$powerEstimate
# adjacency <- adjacency(temp, power = softPower, 
#                        corOptions = list(use = "p", method = "spearman"),
#                       type = "signed hybrid")
# head(adjacency)

### blockwiseModules method

In [ ]:
allowWGCNAThreads()

temp <- pro_data_val
softPower <- sft$powerEstimate
print(str_c("Using power : ",softPower))
net <- blockwiseModules(
    temp,
    power = softPower,
    TOMType = "unsigned",
    corType = "bicor",
    minModuleSize = max(c(20, round(ncol(temp)/50, digits=0))),
    reassignThreshold = 0,
    mergeCutHeight = 0.25,
    numericLabels = TRUE,
    pamRespectsDendro = FALSE,
    saveTOMs = TRUE,
    saveTOMFileBase = "TOM260130",
    verdose = 3
)

moduleColors = labels2colors(net$colors)

plotDendroAndColors(
  net$dendrograms[[1]], 
  moduleColors[net$blockGenes[[1]]],
  "Module colors",
  dendroLabels = FALSE, hang = 0.03,
  addGuide = TRUE, guideHang = 0.05
)

In [ ]:
moduleColors <- labels2colors(net$colors)

table(moduleColors)

## manual

In [ ]:
# # library(WGCNA)
# temp <- pro_data_val

# moduleLabels <- net$colors
# moduleColors <- labels2colors(moduleLabels)

# softPower <- sft$powerEstimate
# adjacency <- adjacency(temp, 
#                        power = softPower, 
#                        type = "signed hybrid",
#                        corFnc = "bicor",
#                        corOptions = list(use = "p"))
# head(adjacency)

In [ ]:
# library(WGCNA)
temp <- pro_data_val

moduleLabels <- net$colors
moduleColors <- labels2colors(moduleLabels)

softPower <- sft$powerEstimate
adjacency <- adjacency(temp, 
                       power = softPower, 
                       type = "unsigned",
                       corFnc = "bicor",
                       corOptions = list(use = "p"))
head(adjacency)

In [ ]:
TOM <- TOMsimilarity(adjacency, TOMType = "unsigned")
dissTOM <- 1-TOM

proteinTree <- hclust(as.dist(dissTOM), method = "average")

options(repr.plot.width = 12, repr.plot.height = 6)
plot(proteinTree, xlab = "", sub = "", main="protein clustering on TOM-based dissimilarity", labels = FALSE, hang=0.04)

In [ ]:
minModuleSize <- max(c(20, round(ncol(data)/50, digits=0)))
print(str_c("minClusterSize = ", minModuleSize))

dynamicMods <- cutreeDynamic(
    dendro=proteinTree, 
    distM=dissTOM,
    deepSplit=4,
    pamStage=TRUE,
    pamRespectsDendro=FALSE,
    minClusterSize=minModuleSize
)
table(dynamicMods)

In [ ]:
dynamicColors <- labels2colors(dynamicMods)
table(dynamicColors)

options(repr.plot.width=12, repr.plot.height=6)
plotDendroAndColors(proteinTree, dynamicColors, "Dynamic Tree Cut",
                    dendroLabels=FALSE, hang=0.03,
                    addGuide=TRUE, guideHang=0.05,
                    main="protein dendrogram and module colors")

In [ ]:
table(net$colors)
table(labels2colors(net$colors))
table(dynamicMods)
dynamicColors <- labels2colors(dynamicMods)

table(dynamicColors)

### Merge

In [ ]:
#Call an automatic merging function
MEDissThres <- 0.2
merge <- mergeCloseModules(pro_data_val, dynamicColors, cutHeight=MEDissThres, verbose=3)

#Eigengenes of the new merged modules
mergedMEs <- merge$newMEs
print(str_c("nrow: ", nrow(mergedMEs)))
head(mergedMEs)

#Update the exparained variance
MEList <- moduleEigengenes(pro_data_val, colors=merge$colors, impute=TRUE, nPC=2)
temp_df <- MEList$varExplained
rownames(temp_df) <- c("PC1", "PC2")
colnames(temp_df) <- str_replace(names(MEList$eigengenes), "^ME", "")
t(temp_df)

#Update the module eigengene clustering
##Calculate dissimilarity of module eigengenes
MEDiss <- 1 - cor(mergedMEs, use="pairwise.complete.obs")
##Cluster module eigengenes
METree <- hclust(as.dist(MEDiss), method="average")
##Plot the result
options(repr.plot.width=10, repr.plot.height=5)
plot(METree, main="Clustering of module eigengenes",
     xlab="", sub="")

In [ ]:
#The merged module colors
mergedColors <- merge$colors
table(mergedColors)

#Plot the dendrogram and module colors
options(repr.plot.width=12, repr.plot.height=6)
plotDendroAndColors(proteinTree, cbind(dynamicColors, mergedColors),
                    c("Dynamic Tree Cut", "Merged dynamic"),
                    dendroLabels=FALSE, hang=0.03,
                    addGuide=TRUE, guideHang=0.05,
                    main="protein dendrogram and module colors")

In [ ]:
#Rename
moduleColors <- mergedColors
MEs <- mergedMEs

#Construct numerical labels corresponding to the colors
colorOrder <- c("grey", standardColors(50))
moduleLabels <- match(moduleColors, colorOrder)-1

In [ ]:
head(pro_analyte)
head(pro_data_val)

In [ ]:
sum(duplicated(colnames(pro_data_val)))

In [ ]:
module_tbl <- tibble(
    Symbol = colnames(pro_data_val),
    ModuleNumber = dynamicMods,
    ModuleID = str_to_title(mergedColors)
)

print(str_c("nrow: ", nrow(module_tbl)))
head(module_tbl)

In [ ]:
# module_tbl
sum(duplicated(module_tbl$Symbol))
table(module_tbl$Symbol)[table(module_tbl$Symbol) > 1]

# pro_analyte
sum(duplicated(pro_analyte$Symbol))
table(pro_analyte$Symbol)[table(pro_analyte$Symbol) > 1]

In [ ]:
#Clean the module eigengenes table
temp_df <- MEs %>%
    rownames_to_column(var="SampleID")
names(temp_df)[2:ncol(temp_df)] <- names(temp_df)[2:ncol(temp_df)] %>%
    str_replace(., "^ME", "") %>%
    str_to_title(.)
print(str_c("nrow: ", nrow(temp_df)))
head(temp_df)
    
#Save module eigengenes values
write.csv(temp_df, file = "./Data/module-eigengenes.csv", row.names = TRUE)

## Calculate eigengenes (Manualモジュールを使って)

In [ ]:
table(dynamicColors)
#Calculate eigengenes
MEList <- moduleEigengenes(pro_data_val, colors=mergedColors, impute=TRUE, nPC=2)
MEs <- MEList$eigengenes
print(str_c("nrow: ", nrow(MEs)))
head(MEs)

#Check the explained variance
temp_df <- MEList$varExplained
rownames(temp_df) <- c("PC1", "PC2")
colnames(temp_df) <- str_replace(names(MEList$eigengenes), "^ME", "")
t(temp_df)

#Calculate dissimilarity of module eigengenes
MEDiss <- 1 - cor(MEs, use="pairwise.complete.obs")

#Cluster module eigengenes
METree <- hclust(as.dist(MEDiss), method="average")

#Plot the result
options(repr.plot.width=10, repr.plot.height=5)
plot(METree, main="Clustering of module eigengenes",
     xlab="", sub="")
# MEDissThres <- 0.2
abline(h=MEDissThres, col="red")

## Merge modules

In [ ]:
mergeCutHeight <- 0.25
merge <- mergeCloseModules(
    temp,
    dynamicColors,
    cutHeight = mergeCutHeight,
    verbose = 3
)
mergedColors <- merge$colors
mergedMEs <- merge$newMEs

table(mergedColors)


table(dynamicColors)

In [ ]:
idx <- match(rownames(MEs), pro_sample$Sample.ID)

all(pro_sample$Sample.ID[idx] == rownames(MEs))

ME_df <- cbind(
    pro_sample[idx, ],
    MEs
)
head(ME_df)

In [ ]:
write.csv(
    ME_df,
    "./Data/Eigengene_ME.csv",
    row.names = FALSE
)
write.csv(
    module_tbl,
    "./Data/WGCNA_module_assignment.csv",
    row.names = FALSE
)

In [ ]:
head(pro_sample)
dim(pro_sample)

In [ ]:

# ME列だけをlong formatに変換
ME_long <- ME_df %>%
    select(CaCo_Status, starts_with("ME")) %>%
    pivot_longer(
        cols = starts_with("ME"),
        names_to = "Module",
        values_to = "Eigengene"
    )

# 全モジュールを一括表示
ggplot(ME_long, aes(x = factor(CaCo_Status), y = Eigengene)) +
    geom_boxplot() +
    facet_wrap(~ Module, scales = "free_y") +
    labs(
        x = "Cancer status",
        y = "Module eigengene (PC1)"
    ) +
    theme_bw()
ME_stats <- ME_long %>%
    group_by(Module) %>%
    summarise(
        p_value = wilcox.test(Eigengene ~ CaCo_Status)$p.value,
        .groups = "drop"
    ) %>%
    arrange(p_value)

ME_stats

## MM

In [ ]:
#Calculate the numbers of Proteins and samples
nproteins <- ncol(pro_data_val)
nSamples <- nrow(pro_data_val)

#Names (colors) of the modules
modNames = substring(names(MEs), 3)

#Calculate module membership
proteinModuleMembership <- as.data.frame(cor(pro_data_val, MEs, use="p"))
#names(proteinModuleMembership) <- paste("MM", modNames, sep="")
names(proteinModuleMembership) <- str_to_title(modNames)
print(str_c("nrow: ", nrow(proteinModuleMembership)))
head(proteinModuleMembership)

#Calculate statisitcal significance of module membership
MMPvalue <- as.data.frame(corPvalueStudent(as.matrix(proteinModuleMembership), nSamples))
#names(MMPvalue) <- paste("p.MM", modNames, sep="")
names(MMPvalue) <- str_to_title(modNames)
print(str_c("nrow: ", nrow(MMPvalue)))
head(MMPvalue)
summary(MMPvalue)

In [ ]:
top_MM_proteins <- lapply(colnames(proteinModuleMembership), function(mod) {
    
    data.frame(
        Symbol = rownames(proteinModuleMembership),
        MM = proteinModuleMembership[[mod]],
        MM_P = MMPvalue[[mod]]
    ) %>%
        arrange(desc(abs(MM))) %>%
        slice_head(n = 10) %>%
        mutate(Module = mod) %>%
        select(Module, Symbol, MM, MM_P)
    
}) %>%
    bind_rows()

View(top_MM_proteins)

# Relationship between each analyte and module

## Module EigengeneとCancerStatusの関連を調べる

In [ ]:
ME_names <- colnames(MEs)

ME_stats <- lapply(ME_names, function(me) {
    
    test <- wilcox.test(
        ME_df[[me]] ~ ME_df$CaCo_Status
    )
    
    data.frame(
        Module = me,
        P_value = test$p.value,
        Cancer_Median = median(
            ME_df[[me]][ME_df$CaCo_Status == 1],
            na.rm = TRUE
        ),
        NonCancer_Median = median(
            ME_df[[me]][ME_df$CaCo_Status == 0],
            na.rm = TRUE
        )
    )
}) %>%
    bind_rows() %>%
    mutate(
        FDR = p.adjust(P_value, method = "BH")
    ) %>%
    arrange(FDR)

ME_stats

## DEPlistのものがどこに入っているか

In [ ]:
head(module_tbl)
head(proteinModuleMembership)

In [ ]:
DEP_module <- module_tbl %>%
    filter(Symbol %in% rownames(DEPlist))



DEP_module_MM <- DEP_module %>%
    rowwise() %>%
    mutate(
        MM = proteinModuleMembership[Symbol, ModuleID],
        MM_P = MMPvalue[Symbol, ModuleID]
    ) %>%
    ungroup() %>%
    mutate(
        DEP_order = match(Symbol, rownames(DEPlist))
    ) %>%
    arrange(DEP_order) %>%
    select(-DEP_order)

DEP_module_MM

In [ ]:
DEP_by_module <- module_tbl %>%
    group_by(ModuleID) %>%
    summarise(
        Total_Proteins = n(),
        DEP_Proteins = sum(DEP),
        DEP_Percent = 100 * DEP_Proteins / Total_Proteins,
        .groups = "drop"
    ) %>%
    arrange(desc(DEP_Percent))

DEP_by_module

In [ ]:
DEP_enrichment <- lapply(
    unique(module_tbl$ModuleID),
    function(mod) {
        
        x <- module_tbl$ModuleID == mod
        y <- module_tbl$DEP
        
        tbl <- table(x, y)
        
        test <- fisher.test(tbl)
        
        data.frame(
            Module = mod,
            Odds_Ratio = unname(test$estimate),
            P_value = test$p.value
        )
    }
) %>%
    bind_rows() %>%
    mutate(
        FDR = p.adjust(P_value, method = "BH")
    ) %>%
    arrange(FDR)

DEP_enrichment

## kME

In [ ]:
#Calculate the numbers of Proteins and samples
nproteins <- ncol(pro_data_val)
nSamples <- nrow(pro_data_val)

#Names (colors) of the modules
modNames = substring(names(MEs), 3)

#Calculate module membership
proteinModuleMembership <- as.data.frame(cor(pro_data_val, MEs, use="p"))
#names(proteinModuleMembership) <- paste("MM", modNames, sep="")
names(proteinModuleMembership) <- str_to_title(modNames)
print(str_c("nrow: ", nrow(proteinModuleMembership)))
head(proteinModuleMembership)

#Calculate statisitcal significance of module membership
MMPvalue <- as.data.frame(corPvalueStudent(as.matrix(proteinModuleMembership), nSamples))
#names(MMPvalue) <- paste("p.MM", modNames, sep="")
names(MMPvalue) <- str_to_title(modNames)
print(str_c("nrow: ", nrow(MMPvalue)))
head(MMPvalue)
summary(MMPvalue)

In [ ]:
#Prepare target modules
targets <- modNames[modNames!="grey"]

#Repeat for each module
temp_tbl <- tibble()
for (module in targets) {
    #Select module probes
    probes <- colnames(pro_data_val)
    inModule <- (moduleColors==module)
    modProbes <- probes[inModule]
    
    #Select the corresponding Topological Overlap
    modTOM <- TOM[inModule, inModule]
    dimnames(modTOM) <- list(modProbes, modProbes)
    
    #Calculate intramodular connectivity
    IMConn <- softConnectivity(pro_data_val[, modProbes], power=softPower,
                               corOptions=list(use="p", method="spearman"), type="signed hybrid")

    proteinname <- pro_analyte%>%
        select(Symbol) %>%
        tibble::rownames_to_column(var = "OlinkID")
    #Summary table
    connectivity <- tibble(
        ModuleID=str_to_title(module),
        OlinkID=modProbes,
        IntramodularConnectivity=IMConn,
        TOMsimilaritySum=rowSums(modTOM))%>%
        dplyr::left_join(proteinname %>% select(OlinkID, Symbol), by = "OlinkID") %>%
        dplyr::select(ModuleID, Symbol, OlinkID, everything())
    
    print(str_c(module," module: ", nrow(connectivity)))
    print(head(connectivity))#Due to within for-loop
    
    #Add to the overall table
    temp_tbl <- dplyr::bind_rows(temp_tbl, connectivity)
}

print(str_c("Total nrow: ", nrow(temp_tbl)))
head(temp_tbl)

In [ ]:
#Update the module assignment table
temp <- dplyr::left_join(module_tbl, temp_tbl, by=c("OlinkID", "ModuleID","Symbol"))
print(str_c("nrow: ", nrow(temp)))
head(temp)
temp %>% dplyr::filter(is.na(IntramodularConnectivity))
module_tbl <- temp
#Save module assignments
write.csv(temp, file = "Data/module-assignments.csv", row.names = TRUE)

In [ ]:
#Check top15 analytes
for (module in targets) {
    print(module)
    module_tbl %>%
        select(-ModuleNumber) %>%
        dplyr::filter(ModuleID==str_to_title(module)) %>%
        dplyr::arrange(desc(IntramodularConnectivity)) %>%
        head(., n=15) %>%
        print()#Due to within for-loop
}

In [ ]:
head(pro_analyte)
head(pro_data_val)

In [ ]:
#Repeat for each module

temp <- pro_analyte %>%
    tibble::rownames_to_column(var = "OlinkID")

for (module in targets) {
    #Select module probes
    probes <- colnames(pro_data_val)
    inModule <- (moduleColors==module)
    modProbes <- probes[inModule]
    print(str_c(module," module: ", length(modProbes)))
    modProteins <- temp$Symbol[match(modProbes, temp$OlinkID)]

    
    #Select the corresponding Topological Overlap
    modTOM <- TOM[inModule, inModule]
    dimnames(modTOM) <- list(modProbes, modProbes)
    
    #Export the network into edge and node list files Cytoscape can read
    fileDir <- "./WGCNA_module/"
    ipynbName <- "WGCNA"
    fileName <- str_c(module)
    cyt <- exportNetworkToCytoscape(modTOM,
                                    edgeFile=str_c(fileDir,fileName,"-edges.txt"),
                                    nodeFile=str_c(fileDir,fileName,"-nodes.txt"),
                                    weighted=TRUE,
                                    threshold=0,
                                    nodeNames=modProbes,
                                    altNodeNames=modProteins,
                                    nodeAttr=str_to_title(moduleColors[inModule]))
}